In [1]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
with open("bilstm_config.json", "r") as f:
    config = json.load(f)
print("Loaded config:", config)


Loaded config: {'embedding_dim': 128, 'hidden_dim': 256, 'output_dim': 3, 'num_layers': 2, 'dropout': 0.5, 'bidirectional': True, 'optimizer': 'Adam', 'learning_rate': 0.001, 'batch_size': 64, 'num_epochs': 10}


In [4]:
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 num_layers, dropout, bidirectional):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
            batch_first=True
        )
        fc_in = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(fc_in, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(emb)
        last = lstm_out[:, -1, :]
        return self.fc(self.dropout(last))


In [5]:
device = torch.device("cpu")
state = torch.load("bilstm_model.pth", map_location="cpu")

# infer dimensions
vocab_size, emb_dim = state["embedding.weight"].shape
hidden_dim = state["lstm.weight_ih_l0"].shape[0] // 4
bidirectional = any(k.startswith("lstm.weight_ih_l0_reverse") for k in state)
num_layers   = 1 + sum(1 for k in state if "lstm.weight_ih_l1" in k)
output_dim   = state["fc.weight"].shape[0]
dropout      = config["dropout"]

print(f"Inferred → vocab_size={vocab_size}, emb_dim={emb_dim}, "
      f"hidden_dim={hidden_dim}, layers={num_layers}, bidi={bidirectional}")

# build model on CPU, load state, then send to GPU if available
model = BiLSTMModel(vocab_size, emb_dim, hidden_dim, output_dim,
                    num_layers, dropout, bidirectional)
model.load_state_dict(state)
model.to(device)
model.eval()
print("Model loaded and ready.\n")


Inferred → vocab_size=10000, emb_dim=128, hidden_dim=64, layers=1, bidi=True
Model loaded and ready.



In [6]:
train_df = pd.read_csv("X_train.csv")
test_df  = pd.read_csv("X_test.csv")

# 2) Define your text lists
X_train_text = train_df["clean_tweet"].astype(str).tolist()
X_test_text  = test_df ["clean_tweet"].astype(str).tolist()

print(f"{len(X_train_text)} train texts, {len(X_test_text)} test texts loaded.")

535425 train texts, 88233 test texts loaded.


In [7]:

y_train_df = pd.read_csv("y_train.csv")
y_test_df  = pd.read_csv("y_test.csv")


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1) Tokenization + padding (Keras)
MAX_VOCAB = 10000
MAX_LEN   = 50
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding="post", truncating="post")

# 2) Label encoding (sklearn)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_df["sentiment_consolidated"])
y_test_enc  = le.transform   (y_test_df ["sentiment_consolidated"])

print("Classes order:", list(le.classes_))  
# should be ['negative','neutral','positive']

# 3) Build DataLoaders
batch_size = config["batch_size"]
train_ds = TensorDataset(torch.tensor(X_train_pad, dtype=torch.long),
                         torch.tensor(y_train_enc, dtype=torch.long))
test_ds  = TensorDataset(torch.tensor(X_test_pad,  dtype=torch.long),
                         torch.tensor(y_test_enc,  dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)


Classes order: ['negative', 'neutral', 'positive']


In [9]:
batch_size = config["batch_size"]

train_ds = TensorDataset(
    torch.tensor(X_train_pad, dtype=torch.long),
    torch.tensor(y_train_enc, dtype=torch.long)
)
test_ds  = TensorDataset(
    torch.tensor(X_test_pad, dtype=torch.long),
    torch.tensor(y_test_enc, dtype=torch.long)
)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

print(f"Loaders ready: {len(train_loader)} train / {len(test_loader)} test batches\n")


Loaders ready: 8367 train / 1379 test batches



In [10]:
device = torch.device("cpu")
def evaluate_model(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            out = model(Xb)
            probs = torch.softmax(out, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            y_true.extend(yb.numpy())
            y_pred.extend(preds)
            y_prob.extend(probs)
    y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.array(y_prob)
    acc = accuracy_score(y_true, y_pred)
    f1s = f1_score(y_true, y_pred, average=None)
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr')
    cm  = confusion_matrix(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 (per class): {f1s}")
    print(f"AUC (OVR): {auc:.4f}")
    print("Confusion Matrix:\n", cm)
    return acc, f1s, auc, cm

print("Test Set Baseline")
test_metrics  = evaluate_model(model, test_loader, device)
print("\n Train Set Baseline")
train_metrics = evaluate_model(model, train_loader, device)


Test Set Baseline
Accuracy: 0.8568
F1 (per class): [0.77727825 0.66052959 0.92426479]
AUC (OVR): 0.9170
Confusion Matrix:
 [[11843  2008  1463]
 [ 1698  9292  2437]
 [ 1618  3408 54466]]

 Train Set Baseline
Accuracy: 0.9837
F1 (per class): [0.98905045 0.98332373 0.97853558]
AUC (OVR): 0.9987
Confusion Matrix:
 [[177088    413    974]
 [   716 175776   1983]
 [  1818   2850 173807]]


In [11]:
import torch
# Force PyTorch to think CUDA isn’t available, so it won’t try graph‐capture checks
torch.cuda.is_available = lambda : False


In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [13]:
# ─── Cell 1: Setup for Dropout Sweeps ───────────────────────────────────
from copy import deepcopy
from torch.optim import Adam

# 1) Save original pretrained weights
orig_state = deepcopy(model.state_dict())

# 2) Helper to retrain & evaluate (CPU only)
def train_and_eval(dropout, lr, epochs=3):
    # a) Fresh model on CPU
    m = BiLSTMModel(vocab_size, emb_dim, hidden_dim,
                    output_dim, num_layers, dropout, bidirectional)
    m.load_state_dict(orig_state)
    m.to("cpu")
    optimizer = Adam(m.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # b) Training loop (CPU)
    for _ in range(epochs):
        m.train()
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(m(Xb), yb)
            loss.backward()
            optimizer.step()

    # c) Evaluation on CPU
    return evaluate_model(m, test_loader, "cpu")

# 3) Common LR
lr_base = config["learning_rate"]
results = []  # will collect all runs
print("Setup complete. Ready to sweep dropouts.")


Setup complete. Ready to sweep dropouts.


In [22]:
# ─── Cell 2: Dropout = 0.3 ────────────────────────────────────────────
print("➡️ Dropout = 0.3")
acc_03, f1s_03, auc_03, _ = train_and_eval(dropout=0.3, lr=lr_base)
results.append({
    "dropout": 0.3,
    "accuracy": acc_03,
    "f1_pos": float(f1s_03[2]),
    "f1_neg": float(f1s_03[0]),
    "f1_neu": float(f1s_03[1]),
    "auc": auc_03
})


➡️ Dropout = 0.3
Accuracy: 0.8634
F1 (per class): [0.78643565 0.67998555 0.92839778]
AUC (OVR): 0.9472
Confusion Matrix:
 [[12732  1650   932]
 [ 2069  9411  1947]
 [ 2264  3192 54036]]


In [23]:
# ─── Cell 3: Dropout = 0.5 ────────────────────────────────────────────
print("➡️ Dropout = 0.5")
acc_05, f1s_05, auc_05, _ = train_and_eval(dropout=0.5, lr=lr_base)
results.append({
    "dropout": 0.5,
    "accuracy": acc_05,
    "f1_pos": float(f1s_05[2]),
    "f1_neg": float(f1s_05[0]),
    "f1_neu": float(f1s_05[1]),
    "auc": auc_05
})


➡️ Dropout = 0.5
Accuracy: 0.8579
F1 (per class): [0.7919068  0.66209089 0.92265042]
AUC (OVR): 0.9457
Confusion Matrix:
 [[12779  1458  1077]
 [ 2071  9164  2192]
 [ 2110  3633 53749]]


In [24]:
# ─── Cell 4: Dropout = 0.7 ────────────────────────────────────────────
print("➡️ Dropout = 0.7")
acc_07, f1s_07, auc_07, _ = train_and_eval(dropout=0.7, lr=lr_base)
results.append({
    "dropout": 0.7,
    "accuracy": acc_07,
    "f1_pos": float(f1s_07[2]),
    "f1_neg": float(f1s_07[0]),
    "f1_neu": float(f1s_07[1]),
    "auc": auc_07
})


➡️ Dropout = 0.7
Accuracy: 0.8310
F1 (per class): [0.75184261 0.59713088 0.90743624]
AUC (OVR): 0.9315
Confusion Matrix:
 [[13108  1084  1122]
 [ 3130  7763  2534]
 [ 3317  3727 52448]]


In [25]:
# ─── Cell 5: Summarize Dropout Sweep ─────────────────────────────────
import pandas as pd

print("### Dropout Tuning Results")
df = pd.DataFrame(results)
display(df[["dropout","accuracy","f1_pos","f1_neg","f1_neu","auc"]])


### Dropout Tuning Results


,dropout,accuracy,f1_pos,f1_neg,f1_neu,auc
0,0.3,0.863384,0.928398,0.786436,0.679986,0.947240
1,0.5,0.857865,0.922650,0.791907,0.662091,0.945698
2,0.7,0.830970,0.907436,0.751843,0.597131,0.931459


In [34]:
from copy import deepcopy
from torch.optim import Adam
import torch.nn as nn
import pandas as pd

# 1) Save original weights
orig_state = deepcopy(model.state_dict())

# 2) Helper for retrain & eval
def train_and_eval_lr(lr, epochs=3):
    m = BiLSTMModel(vocab_size, emb_dim, hidden_dim,
                    output_dim, num_layers, dropout, bidirectional)
    m.load_state_dict(orig_state)
    m.to("cpu")                       # CPU only
    optimizer = Adam(m.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for _ in range(epochs):
        m.train()
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(m(Xb), yb)
            loss.backward()
            optimizer.step()
    
    return evaluate_model(m, test_loader, "cpu")

# 3) Run the trimmed sweep (2 learning rates)
lrs      = [5e-4, config["learning_rate"]]  # only two values: 5e-4 and 1e-3
results_lr = []

for lr in lrs:
    print(f"\n➡️ Learning Rate = {lr}")
    acc, f1s, auc, _ = train_and_eval_lr(lr)
    results_lr.append({
        "lr": lr,
        "accuracy": acc,
        "f1_pos": float(f1s[2]),
        "f1_neg": float(f1s[0]),
        "f1_neu": float(f1s[1]),
        "auc": auc
    })

# 4) Summarize
print("\n### Learning Rate Tuning Results")
print(pd.DataFrame(results_lr))



➡️ Learning Rate = 0.0005
Accuracy: 0.8450
F1 (per class): [0.76497162 0.63931924 0.9154915 ]
AUC (OVR): 0.9403
Confusion Matrix:
 [[13208  1049  1057]
 [ 2845  8377  2205]
 [ 3165  3353 52974]]

➡️ Learning Rate = 0.001
Accuracy: 0.8552
F1 (per class): [0.78085389 0.66162106 0.92182143]
AUC (OVR): 0.9452
Confusion Matrix:
 [[13214  1171   929]
 [ 2549  8873  2005]
 [ 2768  3351 53373]]

### Learning Rate Tuning Results
       lr  accuracy    f1_pos    f1_neg    f1_neu       auc
0  0.0005  0.845024  0.915491  0.764972  0.639319  0.940251
1  0.0010  0.855236  0.921821  0.780854  0.661621  0.945211


In [13]:
from copy import deepcopy
from torch.optim import Adam
import torch.nn as nn
import pandas as pd

# 1) Save the original pretrained weights
orig_state = deepcopy(model.state_dict())

# 2) Helper to retrain & eval with different hidden sizes
def train_and_eval_hidden(hidden_size, epochs=3):
    m = BiLSTMModel(vocab_size, emb_dim, hidden_size,
                    output_dim, num_layers, dropout, bidirectional)
    m.load_state_dict(orig_state, strict=False)
    m.to("cpu")
    optimizer = Adam(m.parameters(), lr=config["learning_rate"])
    criterion = nn.CrossEntropyLoss()
    
    for _ in range(epochs):
        m.train()
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(m(Xb), yb)
            loss.backward()
            optimizer.step()
    
    return evaluate_model(m, test_loader, "cpu")

# 3) Run the trimmed sweep 
hidden_dims = [hidden_dim]  # baseline (64)
results_hd = []

for h in hidden_dims:
    print(f"\n➡️ Hidden Dim = {h}")
    acc, f1s, auc, _ = train_and_eval_hidden(h)
    results_hd.append({
        "hidden_dim": h,
        "accuracy": acc,
        "f1_pos": float(f1s[2]),
        "f1_neg": float(f1s[0]),
        "f1_neu": float(f1s[1]),
        "auc": auc
    })

# 4) Summarize
print("\n### Hidden-Dimension Tuning Results")
print(pd.DataFrame(results_hd))



➡️ Hidden Dim = 64
Accuracy: 0.8533
F1 (per class): [0.77654781 0.65954685 0.92048903]
AUC (OVR): 0.9445
Confusion Matrix:
 [[13013  1328   973]
 [ 2396  8893  2138]
 [ 2792  3319 53381]]

### Hidden-Dimension Tuning Results
   hidden_dim  accuracy    f1_pos    f1_neg    f1_neu       auc
0          64  0.853275  0.920489  0.776548  0.659547  0.944489


In [17]:
import torch

# Save only the weights with a .keras extension
torch.save(model.state_dict(), "bilstm_tuned.keras")
print("✅ Weights saved to bilstm_tuned.keras")

# Or save the full model object
torch.save(model, "bilstm_full_model.keras")
print("✅ Full model saved to bilstm_full_model.keras")


✅ Weights saved to bilstm_tuned.keras
✅ Full model saved to bilstm_full_model.keras


Chat Gpt First Prompt:
Help me to solve cuda error and tell me how many parameter we need to use for empirical tuning.

Chat Gpt Last Prompt:
I had save model in .pth extension so how can I change in .keras format because that will go for interpretability.